# RSNA Knee — train fold 0

로컬에서 수정 → `bash scripts/push_kernel.sh train_fold0` → Kaggle GPU에서 실행.

**푸시 전 체크리스트**
1. `DEBUG = True`로 스모크 테스트 통과
2. `config.verify_labels()` 통과
3. `enable_internet: false` 확인


In [ ]:
# --- 부트스트랩: 로컬/Kaggle 어디서든 src 모듈을 찾게 한다 ---
import os, sys

if os.path.exists('/kaggle/input'):
    # scripts/push_src.sh 로 업로드한 코드 데이터셋
    SRC = '/kaggle/input/knee-src'
else:
    SRC = os.path.abspath(os.path.join('..', '..', 'src'))

sys.path.insert(0, SRC)

import paths, config
print(paths.describe())


In [ ]:
import numpy as np
import pandas as pd

DEBUG = True   # ← 푸시 전 스모크 테스트용. 실제 학습 시 False

cfg = config.CFG(fold=0, debug=DEBUG)
cfg


In [ ]:
# 재현성
import random

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

seed_everything(cfg.seed)


## 데이터 로드


In [ ]:
train = pd.read_csv(paths.TRAIN_CSV)
sub = pd.read_csv(paths.SAMPLE_SUBMISSION_CSV)

config.verify_labels()   # 레이블 순서 검증 - 여기서 막히면 scripts/sync_labels.py 실행

if cfg.debug:
    train = train.head(cfg.debug_samples).reset_index(drop=True)

print(train.shape, sub.shape)
train.head()


## 폴드 분할

TODO: 환자 단위 누수 방지 — `StratifiedGroupKFold(groups=patient_id)` 권장.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold  # noqa: F401

# TODO: 실제 컬럼명 확인 후 작성
# skf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
# train['fold'] = -1
# for f, (_, vi) in enumerate(skf.split(train, train[config.LABELS[0]], train['patient_id'])):
#     train.loc[vi, 'fold'] = f


## Dataset / Model / Train loop

TODO: DICOM 로딩(pydicom) → 전처리 → timm 백본 → BCEWithLogitsLoss 멀티라벨.


In [ ]:
# TODO


## 추론 & 제출

제출 파일은 반드시 `config.make_submission()`으로 만든다 (컬럼 순서 강제).


In [ ]:
# 자리표시자 — 실제 예측으로 교체할 것
test_ids = sub[config.ID_COL].values
probs = np.full((len(test_ids), len(config.LABELS)), 0.5, dtype=np.float32)

submission = config.make_submission(test_ids, probs, path=paths.SUBMISSION_CSV)
print(paths.SUBMISSION_CSV)
submission.head()
